# 03 - Data Preparation (Crop-based)

This notebook prepares the dataset for the **second finetuning stage**.
Instead of resizing the full image, each object is detected using YOLOv8
and cropped individually. This allows the model to focus on the actual objects.

**Run this notebook once before the second training stage.**

```
Original sharp images
        ↓
  YOLOv8 object detection (all objects per image)
        ↓
  Crop each detected object (256×256)
        ↓
  Apply synthetic blur
        ↓
  Save blur/sharp crop pairs → ready for training
```

> One image with 3 objects → 3 crop pairs
> One image with 5 objects → 5 crop pairs

## (Optional) Mount Google Drive

Run this cell only if you are using **Google Colab** and your dataset is stored on Google Drive.
Skip this cell if you are running locally.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

## Step 1 - Install Dependencies

In [ ]:
!pip install -q ultralytics
print('Ultralytics installed. Version:', ultralytics.__version__)

## Step 2 - Set Paths

```
# Google Colab + Drive example:
# SHARP_ROOT  = '/content/drive/MyDrive/your_project/dataset'
# OUTPUT_DIR  = '/content/drive/MyDrive/your_project/dataset_crop'

# Local machine example:
# SHARP_ROOT  = '/path/to/your/dataset'
# OUTPUT_DIR  = '/path/to/your/dataset_crop'
```

The `SHARP_ROOT` folder must contain `train/`, `val/`, and `test/` subfolders with sharp images.

In [ ]:
# ← SET YOUR PATHS HERE
SHARP_ROOT = 'dataset_path'   # folder containing train/ val/ test/ subfolders
OUTPUT_DIR = 'output_path'    # output folder where crop pairs will be saved

from pathlib import Path

for split in ['train', 'val', 'test']:
    p = Path(SHARP_ROOT) / split
    if p.exists():
        n = len([f for f in p.iterdir() if f.suffix.lower() in ('.png','.jpg','.jpeg')])
        print(f'{split:6s}: {n} images found')
    else:
        print(f'{split:6s}: NOT FOUND - {p}')

## Step 3 - Crop Settings

| Parameter | Value | Description |
|-----------|-------|-------------|
| `CROP_PADDING` | 100 | Extra pixels around detected object |
| `MIN_CROP_SIZE` | 128 | Minimum crop size in pixels |
| `OUTPUT_SIZE` | 256 | Final saved crop size (256×256) |
| `CONF_THRESH` | 0.05 | YOLO confidence threshold (low = detect more objects) |

> If YOLO does not detect any object in an image, the center of the image is used as fallback.

In [ ]:
# --- CROP SETTINGS ---
CROP_PADDING  = 100   # extra pixels around detected object
MIN_CROP_SIZE = 128   # minimum crop size
OUTPUT_SIZE   = 256   # final saved size (256x256)
CONF_THRESH   = 0.05  # YOLO confidence threshold
# ----------------------

from ultralytics import YOLO
yolo = YOLO('yolov8n.pt')  # lightweight pretrained model, no labeling needed
print('YOLO loaded.')

## Step 4 - Define Crop Functions

All objects detected in an image are cropped individually.
Crops from the same image are saved with a suffix: `image_obj0.jpg`, `image_obj1.jpg`, etc.

In [ ]:
import cv2, numpy as np
from pathlib import Path

def get_all_crops(img, result, padding, min_size):
    """Extract crops for all detected objects in the image."""
    h, w = img.shape[:2]
    crops = []

    if result.boxes is not None and len(result.boxes) > 0:
        for box in result.boxes.xyxy:
            x1, y1, x2, y2 = box.cpu().numpy()
            cx = int((x1 + x2) / 2)
            cy = int((y1 + y2) / 2)
            obj_w = x2 - x1
            obj_h = y2 - y1
            size  = max(int(max(obj_w, obj_h)) + padding * 2, min_size)
            half  = size // 2
            x1c = max(0, cx - half)
            y1c = max(0, cy - half)
            x2c = min(w, x1c + size)
            y2c = min(h, y1c + size)
            if x2c - x1c < size: x1c = max(0, x2c - size)
            if y2c - y1c < size: y1c = max(0, y2c - size)
            crop = img[int(y1c):int(y2c), int(x1c):int(x2c)]
            if crop.size > 0:
                crops.append((crop, int(x1c), int(y1c), int(x2c), int(y2c)))
    else:
        # Fallback: use image center
        cx, cy = w // 2, h // 2
        half = min_size // 2
        x1c = max(0, cx - half)
        y1c = max(0, cy - half)
        x2c = min(w, x1c + min_size)
        y2c = min(h, y1c + min_size)
        crop = img[y1c:y2c, x1c:x2c]
        if crop.size > 0:
            crops.append((crop, x1c, y1c, x2c, y2c))

    return crops

print('Crop functions defined.')

## Step 5 - Detect Objects and Save Crops

Runs YOLOv8 on every image and saves all detected object crops.

> This step may take a few minutes depending on dataset size.

In [ ]:
from tqdm import tqdm
from pathlib import Path
import cv2

def process_split(split_name):
    src_dir   = Path(SHARP_ROOT) / split_name
    sharp_dst = Path(OUTPUT_DIR) / split_name / 'sharp'
    sharp_dst.mkdir(parents=True, exist_ok=True)

    if not src_dir.exists():
        print(f'{split_name}: not found, skipping.')
        return

    imgs = sorted([f for f in src_dir.iterdir()
                   if f.suffix.lower() in ('.png','.jpg','.jpeg')])
    total_crops = 0
    yolo_found  = 0

    for img_path in tqdm(imgs, desc=f'{split_name}'):
        img = cv2.imread(str(img_path))
        if img is None: continue

        results = yolo(img, conf=CONF_THRESH, verbose=False)
        result  = results[0]

        n_boxes = len(result.boxes) if result.boxes is not None else 0
        if n_boxes > 0:
            yolo_found += 1

        crops = get_all_crops(img, result, CROP_PADDING, MIN_CROP_SIZE)

        stem = img_path.stem
        ext  = img_path.suffix

        for i, (crop, x1c, y1c, x2c, y2c) in enumerate(crops):
            crop_resized = cv2.resize(crop, (OUTPUT_SIZE, OUTPUT_SIZE),
                                      interpolation=cv2.INTER_LANCZOS4)
            save_name = f'{stem}_obj{i}{ext}'
            cv2.imwrite(str(sharp_dst / save_name), crop_resized)
            total_crops += 1

    print(f'{split_name}: {len(imgs)} images → {total_crops} crops '
          f'(YOLO detected objects in {yolo_found} images)')

for split in ['train', 'val', 'test']:
    process_split(split)

print('\nCrop extraction complete.')
print(f'Saved to: {OUTPUT_DIR}/<split>/sharp/')

## Step 6 - Apply Synthetic Blur to Crops

Blur is applied to each crop individually.
Since objects now occupy most of the crop area, blur effects are much more pronounced.

In [ ]:
import cv2, numpy as np, random
from pathlib import Path
from tqdm import tqdm

# --- BLUR SETTINGS ---
GAUSSIAN_SIGMA_RANGE = (2.0, 5.0)
MOTION_KERNEL_RANGE  = (9, 25)
BLUR_MIX_PROB        = 0.5
# ----------------------

def apply_gaussian_blur(img):
    sigma = random.uniform(*GAUSSIAN_SIGMA_RANGE)
    k = int(2 * round(3 * sigma) + 1)
    if k % 2 == 0: k += 1
    return cv2.GaussianBlur(img, (k, k), sigma)

def apply_motion_blur(img):
    k = random.choice(range(MOTION_KERNEL_RANGE[0], MOTION_KERNEL_RANGE[1]+1, 2))
    angle = random.uniform(0, 360)
    kernel = np.zeros((k, k))
    kernel[k//2, :] = 1.0 / k
    M = cv2.getRotationMatrix2D((k//2, k//2), angle, 1)
    kernel = cv2.warpAffine(kernel, M, (k, k))
    kernel /= kernel.sum() + 1e-8
    return cv2.filter2D(img, -1, kernel)

def generate_blur(img):
    return apply_gaussian_blur(img) if random.random() < BLUR_MIX_PROB else apply_motion_blur(img)

for split in ['train', 'val', 'test']:
    sharp_dir = Path(OUTPUT_DIR) / split / 'sharp'
    blur_dir  = Path(OUTPUT_DIR) / split / 'blur'
    if not sharp_dir.exists():
        print(f'{split}: sharp folder not found, skipping.')
        continue
    blur_dir.mkdir(parents=True, exist_ok=True)
    imgs = sorted(sharp_dir.glob('*'))
    for p in tqdm(imgs, desc=f'{split} blur'):
        img = cv2.imread(str(p))
        if img is None: continue
        cv2.imwrite(str(blur_dir / p.name), generate_blur(img))
    print(f'{split}: {len(imgs)} blur crops saved → {blur_dir}')

## Dataset Ready

The crop-based dataset is now prepared with the following structure:
```
OUTPUT_DIR/
  train/
    sharp/   ← object crops (256×256)
    blur/    ← blurred object crops
  val/
    sharp/
    blur/
  test/
    sharp/
    blur/
```

File naming: `original_name_obj0.jpg`, `original_name_obj1.jpg`, ...

Proceed to **04_training_crop_inference.ipynb** to start the second finetuning stage.